# 04 - NLP Experiments: Approach A vs B
**ZHAW AI-Applications: Skin Lesion Classifier**

**Block 2: NLP**

Compares two approaches for extracting clinical features from symptom text:

- **Approach A**: Sentence-Transformers (`all-MiniLM-L6-v2`) — dense semantic embeddings
- **Approach B**: Claude API (`claude-sonnet-4-20250514`) — structured feature extraction

**Rubrik requirement: Both approaches must be implemented and compared.**

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CLASS_NAMES, HAM10000_CLASSES, ANTHROPIC_API_KEY
print(f'Anthropic API key set: {bool(ANTHROPIC_API_KEY)}')

## Approach A: Sentence-Transformer Embeddings

In [ ]:
from src.nlp.embeddings import SymptomEmbedder, generate_synthetic_symptom_data

embedder = SymptomEmbedder()

# Test on sample texts
test_texts = [
    'Dark irregular mole on my back, growing for 3 months, multiple colors',
    'Regular round brown mole, stable for years, no symptoms',
    'Bleeding bump on my nose that wont heal, translucent appearance',
    'Rough scaly red patch on hands, itchy, appears after sun exposure',
]

for text in test_texts:
    result = embedder.predict_class(text)
    print(f'Text: {text[:60]}...')
    print(f'  -> {result["predicted_class"]} ({HAM10000_CLASSES[result["predicted_class"]]}) | sim={result["confidence"]:.3f}')
    print()

In [ ]:
# Class similarity heatmap
from src.nlp.embeddings import SYMPTOM_TEMPLATES

classes = CLASS_NAMES
sim_matrix = np.zeros((len(classes), len(classes)))

for i, cls in enumerate(classes):
    for j, template in enumerate(SYMPTOM_TEMPLATES[classes[0]]):
        sims = embedder.similarity_to_classes(SYMPTOM_TEMPLATES[cls][0])
        for k, c2 in enumerate(classes):
            sim_matrix[i][k] = sims[c2]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sim_matrix, xticklabels=classes, yticklabels=classes, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Class Prototype Similarity (Sentence-Transformer)')
plt.tight_layout()
plt.savefig('../docs/screenshots/nlp_similarity_matrix.png', dpi=150)
plt.show()

## Approach B: Claude API Structured Extraction

In [ ]:
from src.nlp.llm_extractor import ClaudeSymptomExtractor

if ANTHROPIC_API_KEY:
    extractor = ClaudeSymptomExtractor()

    test_text = 'I have a dark irregular spot on my back that has been growing rapidly over the past 2 months. It has uneven jagged edges and shows multiple shades of brown and black. It occasionally bleeds and is about 8mm in diameter.'

    features = extractor.extract(test_text)
    print('Extracted features:')
    for k, v in features.items():
        print(f'  {k:25s}: {v}')

    vec = extractor.to_feature_vector(features)
    print(f'\nFeature vector ({len(vec)}d): {vec}')
else:
    print('[!] No ANTHROPIC_API_KEY. Set it in .env to use Approach B.')

## Comparison: Approach A vs B

In [ ]:
from src.nlp.compare import NLPApproachComparator

comparator = NLPApproachComparator(use_llm=bool(ANTHROPIC_API_KEY))
results_df = comparator.run_comparison(n_samples=300)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['acc_mean', 'f1_mean', 'latency_ms_per_sample']
labels  = ['Accuracy', 'F1-Macro', 'Latency (ms/sample)']
approaches = results_df['approach'].tolist()

for i, (metric, label) in enumerate(zip(metrics, labels)):
    vals = results_df[metric].tolist()
    bars = axes[i].bar(range(len(approaches)), vals, color=['#3498db', '#e74c3c'])
    axes[i].set_xticks(range(len(approaches)))
    axes[i].set_xticklabels(['Approach A\n(Sent-Trans)', 'Approach B\n(Claude)'], fontsize=9)
    axes[i].set_title(label)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                     f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('NLP Approach A vs B Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../docs/screenshots/nlp_comparison.png', dpi=150)
plt.show()

print('\nFull comparison table:')
print(results_df[['approach', 'feature_dim', 'acc_mean', 'f1_mean', 'api_cost_usd', 'latency_ms_per_sample']].to_string(index=False))

## Claude Explanation Demo

In [ ]:
from src.nlp.explainer import LesionExplainer, get_fallback_explanation
from src.config import APP_CONFIG

dummy_cv = {
    'label': 'mel', 'label_name': 'Melanoma', 'confidence': 0.68,
    'top_k': [('mel','Melanoma',0.68), ('nv','Melanocytic nevi',0.22), ('bkl','Benign keratosis',0.06)],
    'probabilities': {c: 0.01 for c in CLASS_NAMES}
}
dummy_cv['probabilities']['mel'] = 0.68

if ANTHROPIC_API_KEY:
    explainer = LesionExplainer()
    explanation = explainer.explain(
        cv_result=dummy_cv,
        symptom_text='Growing dark mole on back for 2 months',
    )
    print(explanation)
else:
    print(get_fallback_explanation(dummy_cv, APP_CONFIG['disclaimer']))